# Analyzing U.S. County-Level Health Outcomes
**DSCI 510 – Spring 2026 | University of Southern California**  
**Author: Summer Almansour**

This notebook runs the full analysis pipeline and displays results.  
All functions are imported from `src/` — no code is duplicated here.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

# Import constants from config (no hardcoded paths or URLs)
from config import (
    CHR_FILEPATH, EPA_FILEPATH, TARGET_COL, FEATURE_COLS,
    CORR_HEATMAP_FILE, STATE_CHART_FILE, SCATTER_FILE, CLUSTER_FILE
)

# Import pipeline functions from src/
from src.data_collection import download_chr_data, download_epa_aqi_data, load_chr_csv, load_epa_csv, fetch_cdc_places_api
from src.data_cleaning import run_cleaning_pipeline
from src.analysis import compute_correlations, compute_correlation_matrix, run_kmeans, state_averages, cluster_summary
from src.visualization import plot_correlation_heatmap, plot_state_comparison, plot_scatter_predictors, plot_clusters

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
%matplotlib inline

print('Imports complete.')

## Step 1: Download Data

In [ ]:
download_chr_data()

try:
    download_epa_aqi_data()
    epa_raw = load_epa_csv()
except Exception as e:
    print(f'EPA download failed: {e}. Proceeding without AQI data.')
    epa_raw = None

try:
    cdc_sample = fetch_cdc_places_api(state_abbr='CA', limit=100)
    print(f'CDC PLACES sample: {len(cdc_sample)} records')
    cdc_sample.head()
except Exception as e:
    print(f'CDC API unavailable: {e}')

## Step 2: Clean and Merge Data

In [ ]:
chr_raw = load_chr_csv()
df = run_cleaning_pipeline(chr_raw, epa_raw)
print(f'Final dataset: {len(df)} counties, {len(df.columns)} columns')
df.head()

## Step 3: Correlation Analysis

In [ ]:
corr_with_target = compute_correlations(df)
print('Correlations with Premature Death Rate:')
print(corr_with_target.to_string())

In [ ]:
corr_matrix = compute_correlation_matrix(df)
plot_correlation_heatmap(corr_matrix)

img = mpimg.imread(CORR_HEATMAP_FILE)
plt.figure(figsize=(9,7))
plt.imshow(img)
plt.axis('off')
plt.show()

## Step 4: State-Level Comparison

In [ ]:
state_avg = state_averages(df)
print('Healthiest states:')
print(state_avg.head(5).to_string())
print('\nLeast healthy states:')
print(state_avg.tail(5).to_string())

plot_state_comparison(state_avg)
img = mpimg.imread(STATE_CHART_FILE)
plt.figure(figsize=(12,5))
plt.imshow(img)
plt.axis('off')
plt.show()

## Step 5: Scatter Plots — Key Predictors

In [ ]:
plot_scatter_predictors(df)
img = mpimg.imread(SCATTER_FILE)
plt.figure(figsize=(12,5))
plt.imshow(img)
plt.axis('off')
plt.show()

## Step 6: K-Means Clustering

In [ ]:
cluster_df, kmeans_model, scaler = run_kmeans(df)

summary = cluster_summary(cluster_df)
print('Cluster Summary:')
print(summary.to_string())

plot_clusters(cluster_df)
img = mpimg.imread(CLUSTER_FILE)
plt.figure(figsize=(13,5))
plt.imshow(img)
plt.axis('off')
plt.show()